In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_score, recall_score, f1_score

In [2]:
df = pd.read_csv("titanic.csv")

In [3]:
df.head()

,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,survived
0,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,1
1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,1
2,1,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,0
3,1,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,0
4,1,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   pclass    1309 non-null   int64  
 1   name      1309 non-null   object 
 2   sex       1309 non-null   object 
 3   age       1046 non-null   float64
 4   sibsp     1309 non-null   int64  
 5   parch     1309 non-null   int64  
 6   ticket    1309 non-null   object 
 7   fare      1308 non-null   float64
 8   cabin     295 non-null    object 
 9   embarked  1307 non-null   object 
 10  survived  1309 non-null   int64  
dtypes: float64(2), int64(4), object(5)
memory usage: 112.6+ KB


In [5]:
df = df.drop(["name", "ticket", "cabin"], axis=1)


In [6]:
X = df.drop("survived", axis=1)
y = df["survived"]

In [7]:
df.isnull().sum()

pclass        0
sex           0
age         263
sibsp         0
parch         0
fare          1
embarked      2
survived      0
dtype: int64

In [8]:


Xtrain, Xtest, ytrain, ytest = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [9]:
num_cols = Xtrain.select_dtypes(include=["int64","float64"]).columns

cat_cols = Xtrain.select_dtypes(include=["object"]).columns

print(num_cols)
print(cat_cols)

Index(['pclass', 'age', 'sibsp', 'parch', 'fare'], dtype='object')
Index(['sex', 'embarked'], dtype='object')


In [10]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])
model_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("logistic_regression", LogisticRegression())
])

In [11]:
model_pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['pclass', 'age', 'sibsp', 'parch', 'fare'], dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['sex', 'embarked'], dtype='object'))])),
                ('logistic_regression', LogisticRegression())])

In [12]:
num_imputer = SimpleImputer(strategy="median")

Xtrain[num_cols] = num_imputer.fit_transform(Xtrain[num_cols])

Xtest[num_cols] = num_imputer.transform(Xtest[num_cols])

In [13]:
cat_imputer = SimpleImputer(strategy="most_frequent")

Xtrain[cat_cols] = cat_imputer.fit_transform(Xtrain[cat_cols])

Xtest[cat_cols] = cat_imputer.transform(Xtest[cat_cols])

In [14]:
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

In [15]:
Xtrain_encoded = encoder.fit_transform(Xtrain[cat_cols])

Xtest_encoded = encoder.transform(Xtest[cat_cols])

In [16]:
encoded_train = pd.DataFrame(
    Xtrain_encoded,
    columns=encoder.get_feature_names_out(cat_cols),
    index=Xtrain.index
)

encoded_test = pd.DataFrame(
    Xtest_encoded,
    columns=encoder.get_feature_names_out(cat_cols),
    index=Xtest.index
)

In [17]:
Xtrain = Xtrain.drop(cat_cols, axis=1)

Xtest = Xtest.drop(cat_cols, axis=1)

In [18]:
Xtrain = pd.concat([Xtrain, encoded_train], axis=1)

Xtest = pd.concat([Xtest, encoded_test], axis=1)

In [19]:
scaler = StandardScaler()

Xtrain[num_cols] = scaler.fit_transform(Xtrain[num_cols])

Xtest[num_cols] = scaler.transform(Xtest[num_cols])

In [20]:
model = LogisticRegression(random_state=42)

model.fit(Xtrain, ytrain)

LogisticRegression(random_state=42)

In [21]:
ypred = model.predict(Xtest)

In [22]:
accuracy = accuracy_score(ytest, ypred)

print("Accuracy:", accuracy)

Accuracy: 0.7748091603053435


In [23]:
cm = confusion_matrix(ytest, ypred)

print(cm)

[[127  17]
 [ 42  76]]


In [24]:
precision = precision_score(ytest, ypred)

print("Precision:", precision)

Precision: 0.8172043010752689


In [25]:
recall = recall_score(ytest, ypred)

print("Recall:", recall)

Recall: 0.6440677966101694


In [26]:
f1 = f1_score(ytest, ypred)

print("F1 Score:", f1)

F1 Score: 0.7203791469194313


In [27]:
print(classification_report(ytest, ypred))

              precision    recall  f1-score   support

           0       0.75      0.88      0.81       144
           1       0.82      0.64      0.72       118

    accuracy                           0.77       262
   macro avg       0.78      0.76      0.77       262
weighted avg       0.78      0.77      0.77       262

